# Práctica 6 · Cuando ya no son tres documentos

Todo lo que construimos funciona con un puñado de archivos. Hoy averiguamos qué pasa cuando son
cientos, y qué hay que cambiar.

La respuesta corta, y la vas a comprobar tú mismo, es que escala mejor de lo que uno teme. La
respuesta larga es que aparecen problemas nuevos, y ninguno tiene que ver con la búsqueda: tienen
que ver con el tiempo, con la memoria, y sobre todo con qué haces cuando los documentos cambian.

Esta práctica es más de medición que de código nuevo. La idea es que salgas con números propios
para poder planear, en lugar de con reglas de dedo.

Las celdas se ejecutan en orden, una por una, con Shift + Enter.

In [1]:
%pip install --quiet pymupdf

print("Listo.")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/fgrodriguez/ESAN_GlobalWeek2026/09_Notebooks_RAG/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Listo.


## 2. Comprobar que Ollama responde

Ollama es el programa que ejecuta los modelos de lenguaje dentro de tu computadora. Tiene que
estar encendido para que este cuaderno funcione, así que lo primero es confirmarlo.

Si algo falla, la salida de la celda te dice qué hacer según tu sistema operativo.

In [2]:
import platform
import sys

import requests

OLLAMA_URL = "http://localhost:11434"

print(f"Sistema: {platform.system()} {platform.machine()}")
print(f"Python:  {sys.version.split()[0]}\n")

try:
    respuesta = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    respuesta.raise_for_status()
    modelos = sorted(m["name"] for m in respuesta.json()["models"])
    print(f"Ollama responde. Tienes {len(modelos)} modelos descargados:\n")
    for m in modelos:
        print(f"  - {m}")
except Exception as e:
    print(f"Ollama no responde en {OLLAMA_URL}")
    print(f"Detalle: {type(e).__name__}\n")
    if platform.system() == "Darwin":
        print("En Mac: abre la aplicación Ollama desde la carpeta Aplicaciones.")
        print("Debe aparecer su ícono en la barra de menús, arriba a la derecha.")
    elif platform.system() == "Windows":
        print("En Windows: busca Ollama en el menú Inicio y ábrelo.")
        print("Debe aparecer su ícono junto al reloj, abajo a la derecha.")
    else:
        print("Ejecuta 'ollama serve' en una terminal.")

Sistema: Darwin arm64
Python:  3.12.13

Ollama responde. Tienes 14 modelos descargados:

  - embeddinggemma:300m
  - gemma3:1b
  - gemma3:4b
  - gemma4:12b-mlx
  - gemma4:26b
  - gemma4:e4b
  - granite4.1:3b
  - granite4.1:8b
  - mxbai-embed-large:latest
  - nemotron-mini:4b
  - nomic-embed-text:latest
  - qwen3-4b-cs-ft:latest
  - qwen3:4b
  - shieldgemma:2b


## 3. Elegir el modelo según tu equipo

El modelo va a correr en tu máquina, así que la memoria que tengas importa. Un modelo grande
en un equipo chico no se rompe: simplemente tarda muchísimo y el sistema se pone lento.

Abajo hay tres opciones. Deja activa una sola, la que corresponda a tu computadora, y comenta
las demás poniéndoles un signo de gato al inicio de la línea. Si no sabes cuánta memoria
tienes, quédate con la opción A, que funciona en cualquier equipo.

El modelo de embeddings no se elige por equipo: es ligero y va igual en todos. Sí conviene saber
de dónde salió esa elección, y la respuesta es que está medida con documentos en español; en la
práctica 3 vas a reproducir la medición y a ver a los tres candidatos compitiendo.

In [3]:
# ---- Opción A: equipos de 8 GB de memoria o menos (descarga 3.3 GB) ---------
MODELO_LLM = "gemma3:4b"

# ---- Opción B: equipos de 16 GB de memoria (descarga 10 GB) ----------------
# MODELO_LLM = "gemma4:12b"        # Windows y Linux
# MODELO_LLM = "gemma4:12b-mlx"    # Mac con chip Apple (M1 en adelante), va más rápido

# ---- Opción C: equipos de 32 GB de memoria o más (descarga 17 GB) ----------
# MODELO_LLM = "gemma4:26b"        # Windows y Linux
# MODELO_LLM = "gemma4:26b-mlx"    # Mac con chip Apple

# El modelo de embeddings es ligero y es el mismo para todos. La elección está
# medida, no copiada de un tutorial: lo comprobamos en la práctica 3. Se eligió
# éste porque es el único de los tres que encuentra un pasaje en inglés cuando la
# pregunta va en español, algo que hace falta en cuanto el corpus mezcla idiomas.
MODELO_EMBEDDINGS = "embeddinggemma:300m"

print(f"Modelo de lenguaje:   {MODELO_LLM}")
print(f"Modelo de embeddings: {MODELO_EMBEDDINGS}")
print("\nSi alguno no aparece en la lista de la celda anterior, descárgalo con:")
print(f"   ollama pull {MODELO_LLM}")
print(f"   ollama pull {MODELO_EMBEDDINGS}")

Modelo de lenguaje:   gemma3:4b
Modelo de embeddings: embeddinggemma:300m

Si alguno no aparece en la lista de la celda anterior, descárgalo con:
   ollama pull gemma3:4b
   ollama pull embeddinggemma:300m


## 4. El material de hoy

Necesitamos volumen, así que además de los documentos de TiendaSol usamos los archivos más
grandes que hay en la carpeta. No importa de qué tratan: hoy nos interesan como carga de trabajo.

In [4]:
import os
import warnings

warnings.filterwarnings("ignore", message=".*langchain-community.*")
# LanceDB avisa por consola cada vez que crea una tabla y cuando una API
# cambiará en el futuro. Nada de eso es un problema; lo callamos para que la
# salida de las mediciones se lea limpia.
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["RUST_LOG"] = "error"

import time
from pathlib import Path

import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Hasta aquí veníamos trabajando con un documento a la vez. Una empresa real tiene
# carpetas enteras, y ahí aparecen problemas que con un archivo no se ven.
CARPETA = Path("documentos")
divisor = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# glob("*.pdf") lista todos los archivos que terminan en .pdf. sorted() los pone en
# orden alfabético, para que la salida sea siempre la misma y se pueda comparar.
archivos = sorted(CARPETA.glob("*.pdf"))
print(f"{'archivo':<38} {'páginas':>8} {'fragmentos':>12}")
print("-" * 60)

corpus = []
for ruta in archivos:
    doc = pymupdf.open(ruta)
    texto = " ".join(p.get_text() for p in doc)
    trozos = divisor.split_text(texto)
    corpus += [Document(page_content=t, metadata={"fuente": ruta.name}) for t in trozos]
    print(f"{ruta.name:<38} {len(doc):>8} {len(trozos):>12}")

print("-" * 60)
print(f"{'TOTAL':<38} {'':>8} {len(corpus):>12}")

archivo                                 páginas   fragmentos
------------------------------------------------------------
Alice_in_Wonderland.pdf                     105          430
tiendasol_anexo_envenenado.pdf                1            2
tiendasol_catalogo.pdf                        1            5
tiendasol_politicas.pdf                       2            6
------------------------------------------------------------
TOTAL                                                    443


## 5. Documentos grandes: el problema no es el tamaño, es cargarlo entero

El primer obstáculo con un archivo de miles de páginas no es que tarde, es que no cabe. Cargarlo
completo en memoria para después trocearlo puede tumbar el proceso.

La solución es no cargarlo entero. Se procesa página por página, o por bloques de páginas, y se
suelta lo ya procesado. El consumo de memoria deja de depender del tamaño del archivo.

Aquí están las dos formas, para que veas la diferencia en el pico de memoria.

In [5]:
import gc
import resource


# Cuánta memoria lleva usada el proceso. La usamos antes y después de cada estrategia
# para ver la diferencia; no es una medida exacta, pero basta para comparar.
def memoria_mb():
    uso = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return uso / (1024 * 1024)  # en macOS viene en bytes


# Estrategia 1: leer el documento entero a memoria y luego partirlo. Es la forma
# natural de escribirlo, y funciona bien hasta que el documento es grande.
def cargar_de_golpe(ruta):
    """Todo el texto del documento en una sola cadena."""
    doc = pymupdf.open(ruta)
    texto = " ".join(p.get_text() for p in doc)
    return divisor.split_text(texto)


# Estrategia 2: procesar de 20 páginas en 20 páginas y soltar cada bloque al terminar.
# El resultado es idéntico; lo que cambia es cuánta memoria se ocupa a la vez.
def cargar_por_partes(ruta, paginas_por_bloque=20):
    """Procesa el documento por bloques de páginas y va soltando lo usado."""
    doc = pymupdf.open(ruta)
    trozos = []
    for inicio in range(0, len(doc), paginas_por_bloque):
        bloque = " ".join(doc[i].get_text()
                          for i in range(inicio, min(inicio + paginas_por_bloque, len(doc))))
        trozos += divisor.split_text(bloque)
        del bloque
    return trozos


# La diferencia solo se nota con el documento más grande, así que lo buscamos.
grande = max(archivos, key=lambda r: len(pymupdf.open(r)))
print(f"Documento más grande: {grande.name} ({len(pymupdf.open(grande))} páginas)\n")

gc.collect()
antes = memoria_mb()
a = cargar_de_golpe(grande)
pico_golpe = memoria_mb() - antes

gc.collect()
antes = memoria_mb()
b = cargar_por_partes(grande)
pico_partes = memoria_mb() - antes

print(f"{'estrategia':<22} {'fragmentos':>12} {'memoria extra':>15}")
print("-" * 52)
print(f"{'de golpe':<22} {len(a):>12} {pico_golpe:>13.1f} MB")
print(f"{'por bloques':<22} {len(b):>12} {pico_partes:>13.1f} MB")

Documento más grande: Alice_in_Wonderland.pdf (105 páginas)



estrategia               fragmentos   memoria extra
----------------------------------------------------
de golpe                        430           0.9 MB
por bloques                     433           2.0 MB


Con un documento de cien páginas la diferencia es pequeña y puede incluso salir a favor de
cualquiera de las dos, porque a esta escala el ruido de medición pesa más que el efecto.

El punto no es el número de hoy, es el patrón: el consumo de la primera crece con el tamaño del
archivo y el de la segunda no. Con un manual de diez mil páginas, la primera falla y la segunda
sigue funcionando igual. Y como no cuesta más escribirla, conviene usar la segunda desde el
principio.

Fíjate además en que ambas producen el mismo número de fragmentos. Procesar por partes no cambia
el resultado, solo la forma de llegar a él. Ojo con un detalle: si un fragmento quedara a caballo
entre dos bloques, se partiría distinto. Por eso los bloques se hacen por páginas y no por
caracteres.

## 6. Cuánto tarda vectorizar

Este es el paso caro de toda la ingesta, y el que hay que saber medir para poder prometer plazos.
Vamos a cronometrar con volúmenes crecientes.

In [6]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=MODELO_EMBEDDINGS)
material = (corpus * 4)  # repetimos para tener suficiente carga

print(f"{'fragmentos':>11} {'segundos':>9} {'fragmentos/seg':>16}")
print("-" * 40)
ritmos = []
# Medimos con cantidades crecientes. La primera siempre sale lenta porque el modelo
# se está cargando en memoria, y por eso abajo se descarta para el promedio.
for cuantos in (50, 100, 200, 400):
    muestra = [d.page_content for d in material[:cuantos]]
    inicio = time.time()
    embeddings.embed_documents(muestra)
    tardo = time.time() - inicio
    ritmo = cuantos / tardo
    ritmos.append(ritmo)
    print(f"{cuantos:>11} {tardo:>9.1f} {ritmo:>16.1f}")

# [1:] deja fuera la primera medición. Este es el número que sirve para planear.
estable = sum(ritmos[1:]) / len(ritmos[1:])
print(f"\nRitmo estable: {estable:.0f} fragmentos por segundo")

 fragmentos  segundos   fragmentos/seg
----------------------------------------


         50       1.4             37.0


        100       1.3             76.5


        200       2.5             79.0


        400       5.0             79.2

Ritmo estable: 78 fragmentos por segundo


Lo importante de esa tabla es que el ritmo se mantiene. Vectorizar cuatrocientos fragmentos tarda
cuatro veces lo que cuarenta, no dieciséis veces. El trabajo crece en línea recta con el volumen,
que es la mejor noticia posible para planear.

La primera fila suele salir más lenta que las demás porque incluye la carga del modelo en memoria.
Es un costo fijo que se paga una vez, no por fragmento.

Con ese ritmo ya puedes proyectar.

In [7]:
# Con el ritmo medido se puede responder la pregunta que hace cualquier jefe: ¿cuánto
# va a tardar esto? La respuesta es una multiplicación, no una corazonada.
CASOS = [
    ("un manual de 200 páginas", 800),
    ("un libro de 500 páginas", 2_000),
    ("el centro de ayuda completo", 10_000),
    ("todo el repositorio documental", 100_000),
]

print(f"{'caso':<34} {'fragmentos':>11} {'tiempo estimado':>18}")
print("-" * 66)
for descripcion, cantidad in CASOS:
    segundos = cantidad / estable
    if segundos < 90:
        tiempo = f"{segundos:.0f} s"
    elif segundos < 5400:
        tiempo = f"{segundos/60:.0f} min"
    else:
        tiempo = f"{segundos/3600:.1f} h"
    print(f"{descripcion:<34} {cantidad:>11,} {tiempo:>18}")

caso                                fragmentos    tiempo estimado
------------------------------------------------------------------
un manual de 200 páginas                   800               10 s
un libro de 500 páginas                  2,000               26 s
el centro de ayuda completo             10,000              2 min
todo el repositorio documental         100,000             21 min


Estos números suelen sorprender, y para bien. Un libro entero se indexa en menos de un minuto en
una laptop; el centro de ayuda completo de una tienda, en un par de minutos.

Conviene aterrizar lo que significan. Que un corpus de diez mil fragmentos se procese en minutos
quiere decir que la ingesta no es el problema de un sistema RAG de tamaño mediano. Los proyectos
no se atoran aquí, se atoran en conseguir los documentos, en que estén actualizados y en que
alguien decida qué versión es la buena.

Y ojo con extrapolar demasiado lejos. La proyección a cien mil fragmentos supone que el ritmo se
mantiene, y a esa escala aparecen cosas que no medimos: la memoria del proceso, el tamaño del
índice, la necesidad de reintentar los archivos que fallan.

## 7. No repetir trabajo: huellas de documento

Aquí empieza la parte que de verdad cambia cuando el sistema deja de ser un ejercicio.

Un corpus real no se indexa una vez: se reindexa cada vez que llega documentación nueva. Si cada
reindexado reprocesa todo, el costo crece sin motivo y el sistema queda fuera de servicio
mientras tanto.

La solución es guardar una huella de cada documento, y volver a procesar solo lo que cambió. La
huella se calcula del contenido, no de la fecha del archivo, porque copiar un archivo le cambia
la fecha sin cambiar lo que dice.

In [8]:
import hashlib
import json


# sha256 resume el contenido de un archivo en una cadena corta. Si cambia una sola
# letra, la huella cambia por completo. Se usa el contenido y no la fecha porque
# copiar un archivo le cambia la fecha sin cambiar nada de lo que dice.
def huella(ruta):
    """Huella del contenido del archivo, independiente de su fecha."""
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()[:16]


REGISTRO = Path("registro_ingesta.json")


def cargar_registro():
    return json.loads(REGISTRO.read_text()) if REGISTRO.exists() else {}


def guardar_registro(registro):
    REGISTRO.write_text(json.dumps(registro, indent=2, ensure_ascii=False))


# Reindexar todo cada vez es caro y lento. Esto compara las huellas guardadas contra
# las de ahora y clasifica cada archivo en nuevo, cambiado, igual o borrado. Solo se
# procesan los dos primeros grupos.
def planear_ingesta(carpeta):
    """Decide qué archivos hay que procesar, cuáles saltar y cuáles se borraron."""
    registro = cargar_registro()
    actuales = {r.name: huella(r) for r in sorted(Path(carpeta).glob("*.pdf"))}

    nuevos = [n for n in actuales if n not in registro]
    cambiados = [n for n in actuales if n in registro and registro[n] != actuales[n]]
    iguales = [n for n in actuales if n in registro and registro[n] == actuales[n]]
    borrados = [n for n in registro if n not in actuales]

    return actuales, nuevos, cambiados, iguales, borrados


actuales, nuevos, cambiados, iguales, borrados = planear_ingesta(CARPETA)

print("Primera pasada, sin registro previo:")
print(f"  nuevos:    {len(nuevos)}")
print(f"  cambiados: {len(cambiados)}")
print(f"  iguales:   {len(iguales)}  (estos se saltan)")
print(f"  borrados:  {len(borrados)}")

guardar_registro(actuales)

Primera pasada, sin registro previo:
  nuevos:    0
  cambiados: 0
  iguales:   4  (estos se saltan)
  borrados:  0


Ahora la segunda pasada, sin haber tocado nada. Es la situación normal: alguien dispara la
ingesta y casi todo sigue igual.

In [9]:
actuales, nuevos, cambiados, iguales, borrados = planear_ingesta(CARPETA)

print("Segunda pasada, con el registro ya escrito:")
print(f"  nuevos:    {len(nuevos)}")
print(f"  cambiados: {len(cambiados)}")
print(f"  iguales:   {len(iguales)}  (estos se saltan)")
print(f"  borrados:  {len(borrados)}")

ahorro = len(iguales) / max(len(actuales), 1)
print(f"\nTrabajo evitado: {ahorro:.0%} de los archivos")

Segunda pasada, con el registro ya escrito:
  nuevos:    0
  cambiados: 0
  iguales:   4  (estos se saltan)
  borrados:  0

Trabajo evitado: 100% de los archivos


Esa es la diferencia entre una ingesta que se puede correr cada hora y una que solo se puede
correr de madrugada.

Nota la palabra que describe esta propiedad: idempotencia. Significa que correr el proceso dos
veces deja el sistema igual que correrlo una vez. Suena obvio y casi nunca se cumple por
accidente: sin el registro, la segunda pasada duplicaría cada fragmento en el índice, y las
búsquedas empezarían a devolver el mismo texto dos veces.

Y fíjate en los borrados, que es el caso que más se olvida. Si un documento desaparece de la
carpeta y nadie quita sus fragmentos del índice, el sistema va a seguir respondiendo con una
política que ya no existe. Eso no da error: da respuestas obsoletas, que es peor porque nadie se
entera.

## 8. Actualizar de verdad: borrar lo viejo antes de meter lo nuevo

Con el plan de arriba, la ingesta incremental es corta. Lo único delicado es el orden: primero se
quitan del índice los fragmentos del documento que cambió, y después se insertan los nuevos.

In [10]:
from langchain_community.vectorstores import LanceDB

almacen = LanceDB.from_documents(corpus, embeddings, uri="lancedb_p6",
                                 table_name="escala", mode="overwrite")
tabla = almacen.get_table()
print(f"Índice inicial: {tabla.count_rows()} fragmentos")


# Actualizar un documento no es agregarlo otra vez: hay que quitar primero sus
# fragmentos viejos. Si no, la versión anterior sigue en el índice y el sistema puede
# responder con la política que ya se derogó.
def reindexar_documento(ruta, almacen):
    """Quita los fragmentos anteriores de ese documento y mete los actuales."""
    nombre = Path(ruta).name
    tabla = almacen.get_table()

    antes = tabla.count_rows()
    # Aquí se ve para qué servían los metadatos: permiten borrar justo los fragmentos
    # de un archivo, sin tocar los demás.
    tabla.delete(f"metadata.fuente = '{nombre}'")
    quitados = antes - tabla.count_rows()

    doc = pymupdf.open(ruta)
    trozos = []
    for inicio in range(0, len(doc), 20):
        bloque = " ".join(doc[i].get_text()
                          for i in range(inicio, min(inicio + 20, len(doc))))
        trozos += divisor.split_text(bloque)

    nuevos = [Document(page_content=t, metadata={"fuente": nombre}) for t in trozos]
    almacen.add_documents(nuevos)

    return quitados, len(nuevos)


objetivo = CARPETA / "tiendasol_politicas.pdf"
quitados, agregados = reindexar_documento(objetivo, almacen)

print(f"\nReindexado de {objetivo.name}:")
print(f"  fragmentos retirados: {quitados}")
print(f"  fragmentos añadidos:  {agregados}")
print(f"  total en el índice:   {almacen.get_table().count_rows()}")

Índice inicial: 443 fragmentos



Reindexado de tiendasol_politicas.pdf:
  fragmentos retirados: 6
  fragmentos añadidos:  6
  total en el índice:   6


El total vuelve a ser el mismo, que es justo lo que queremos: reindexar un documento no debe
inflar el índice.

Si al correr esta celda dos veces el número creciera, tendrías el problema de duplicados del que
hablábamos. Vale la pena que la ejecutes otra vez y lo compruebes.

Para que el borrado por documento funcione hay que haber guardado la fuente en los metadatos
desde el principio. Es la segunda vez en el curso que esa decisión de la práctica 2 paga: primero
sirvió para citar la fuente, ahora para poder actualizar sin reconstruir todo.

## 9. ¿Y la búsqueda? ¿Se vuelve lenta?

Queda la duda razonable: si el índice crece a decenas de miles de fragmentos, ¿cada consulta
tendrá que compararse contra todos?

Sí, salvo que construyas un índice especial. Vamos a medir cuánto duele.

In [11]:
import numpy as np
import lancedb

vectores_base = np.array(embeddings.embed_documents(
    [d.page_content for d in corpus[:200]]))
consulta = np.array(embeddings.embed_query("¿cuánto cuesta el envío?"), dtype=np.float32)

db = lancedb.connect("lancedb_escala")

print(f"{'fragmentos':>11} {'exhaustiva':>13} {'con índice':>13}")
print("-" * 40)
# Con pocos fragmentos, comparar la pregunta contra todos es instantáneo. La pregunta
# es a partir de cuántos deja de serlo, y cuánto ayuda entonces un índice.
for objetivo in (1_000, 5_000, 20_000):
    repeticiones = objetivo // len(vectores_base) + 1
    v = np.vstack([vectores_base + np.random.normal(0, 0.01, vectores_base.shape)
                   for _ in range(repeticiones)])[:objetivo]
    v = (v / np.linalg.norm(v, axis=1, keepdims=True)).astype(np.float32)

    tabla = db.create_table(f"prueba_{objetivo}",
                            data=[{"vector": v[i], "id": i} for i in range(objetivo)],
                            mode="overwrite")

    inicio = time.time()
    for _ in range(20):
        tabla.search(consulta).limit(3).to_list()
    exhaustiva = (time.time() - inicio) / 20 * 1000

    # El índice agrupa los vectores en zonas para no revisarlos todos. Gana velocidad
    # a cambio de exactitud: puede dejar fuera algún resultado bueno. Es un cambio que
    # solo vale la pena cuando la búsqueda exhaustiva ya se siente lenta.
    tabla.create_index(num_partitions=max(2, objetivo // 1000), num_sub_vectors=16)
    inicio = time.time()
    for _ in range(20):
        tabla.search(consulta).limit(3).to_list()
    con_indice = (time.time() - inicio) / 20 * 1000

    print(f"{objetivo:>11,} {exhaustiva:>11.1f} ms {con_indice:>11.1f} ms")

 fragmentos    exhaustiva    con índice
----------------------------------------
      1,000         2.2 ms         1.4 ms


[2026-08-07T18:44:19Z WARN  lance_index::vector::kmeans] KMeans: more than 10% of clusters are empty: 1 of 2.
    Help: this could mean your dataset has many duplicate vectors.


[2026-08-07T18:44:19Z WARN  lance_index::vector::kmeans] KMeans: more than 10% of clusters are empty: 1 of 5.
    Help: this could mean your dataset has many duplicate vectors.


      5,000         2.8 ms         1.2 ms


     20,000         5.4 ms         1.8 ms


La búsqueda exhaustiva crece con el tamaño del índice, como era de esperar, mientras que con el
índice construido el tiempo se mantiene plano.

Pero mira las cifras absolutas antes de correr a construir índices. Con veinte mil fragmentos, la
búsqueda exhaustiva tarda unos pocos milisegundos. En una consulta que además incluye vectorizar
la pregunta y esperar a que el modelo redacte, esos milisegundos no se notan: el modelo se tarda
mil veces más.

O sea que para el tamaño de corpus de la mayoría de las empresas medianas, unas decenas de miles
de fragmentos, la búsqueda exhaustiva alcanza de sobra. El índice aproximado empieza a importar en
cientos de miles o millones, y trae su propio costo: hay que construirlo, ocupa espacio, hay que
reconstruirlo cuando el corpus cambia mucho, y por definición puede devolver un resultado
ligeramente peor que el exacto.

Es el mismo criterio de siempre: no agregues la pieza hasta que la medición diga que hace falta.

## 10. Lo que de verdad se rompe a escala

Cerramos con una lista de lo que sí se vuelve difícil, porque no es lo que uno espera.

El tiempo de vectorizado no es el problema. Ya viste que escala en línea recta y que un corpus
mediano se procesa en minutos.

La velocidad de búsqueda tampoco, hasta escalas grandes.

Lo que se rompe es todo lo demás. Que un archivo de los mil que estás procesando esté corrupto y
tumbe el proceso entero en el número novecientos. Que la ingesta tarde horas y nadie sepa en qué
va. Que se corra dos veces y se dupliquen los fragmentos. Que un documento se elimine y sus
fragmentos sigan contestando. Que nadie sepa qué versión de la política está indexada.

Todos esos problemas son de proceso, no de algoritmos. Por eso a escala la ingesta deja de ser un
script y pasa a ser una tubería con reintentos, registro de lo procesado, y visibilidad de en qué
va. Hoy montaste la pieza central de eso, que es el registro de huellas.

Una última idea, que es la que más me interesa que te lleves. Cuando alguien te diga que su
sistema RAG no escala, pregúntale qué midió. Es probable que no haya medido nada y esté repitiendo
lo que leyó. Tú ya tienes los números de tu propia máquina.